# Week 7 – Outlier Detection and Data Quality

This notebook applies outlier detection to the Week 6 engineered sold dataset. The goal is to flag extreme values in key numeric fields using the Interquartile Range (IQR) method, save a full flagged dataset, create a clean filtered analysis dataset, and compare dataset size and median values before and after filtering.

In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/Users/amyliu/Desktop/IDX")

# Input from Week 6
SOLD_FILE = BASE_DIR / "data" / "generated" / "week6" / "sold_week6_engineered_metrics.csv"
LISTINGS_FILE = BASE_DIR / "data" /"generated" / "week6" / "listing_week6_engineered_metrics.csv"

# Output folder for Week 7
OUTPUT_DIR = BASE_DIR / "data" / "week7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input sold file:", SOLD_FILE)
print("Output folder:", OUTPUT_DIR)

if not SOLD_FILE.exists():
    raise FileNotFoundError(f"Sold file not found: {SOLD_FILE}")

Input sold file: /Users/amyliu/Desktop/IDX/data/generated/week6/sold_week6_engineered_metrics.csv
Output folder: /Users/amyliu/Desktop/IDX/data/week7


## Load Week 6 Engineered Sold Dataset

The Week 6 engineered sold dataset is used as the input for Week 7 outlier detection.

In [22]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)

print(f"Rows loaded: {len(sold):,}")
print(f"Columns loaded: {sold.shape[1]:,}")

sold.head()

Rows loaded: 397,603
Columns loaded: 93


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,ListAgentFirstName,...,implausible_coord_flag,price_ratio,close_to_original_list_ratio,price_per_sqft,days_on_market,close_year,close_month,yrmo,listing_to_contract_days,contract_to_close_days
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,False,499000.0,551985747,2024-01-26,240000.0,Joan,...,False,0.480962,0.480962,210.526316,777,2024,1,2024-01,777.0,65.0
1,SanDiego,SanDiego,NaN,False,False,759900.0,522107581,2024-01-05,815000.0,Michael,...,False,1.072510,1.072510,412.867275,33,2024,1,2024-01,114.0,919.0
2,SanDiego,SanDiego,NaN,False,False,739900.0,510919001,2024-01-05,810000.0,Michael,...,False,1.094743,1.094743,410.334347,228,2024,1,2024-01,255.0,778.0
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,1079166779,2024-01-30,858000.0,David,...,False,NaN,NaN,430.075188,0,2024,1,2024-01,188.0,-188.0
4,Southland,Southland,NaN,False,False,1890500.0,1075037759,2024-01-29,1890500.0,Karen,...,False,1.000000,1.000000,591.891046,0,2024,1,2024-01,0.0,0.0


In [23]:
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

print(f"Rows loaded: {len(listings):,}")
print(f"Columns loaded: {listings.shape[1]:,}")

listings.head()

Rows loaded: 540,183
Columns loaded: 79


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,negative_timeline_flag,missing_coord_flag,zero_coord_flag,positive_longitude_flag,implausible_coord_flag,list_year,list_month,list_yrmo,list_price_per_sqft,listing_days_on_market
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,False,False,False,False,2024,1,2024-01,1029.976941,127
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,False,False,False,False,2024,1,2024-01,896.700143,1
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,False,False,False,False,False,2024,1,2024-01,969.230769,1
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,False,False,False,False,False,2024,1,2024-01,414.431330,0
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,False,False,False,False,False,2024,1,2024-01,5482.550625,2


## Check Key Numeric Fields

The Week 7 deliverable requires outlier detection for ClosePrice, LivingArea, and DaysOnMarket.

In [24]:
outlier_fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

field_check = pd.DataFrame({
    "field": outlier_fields,
    "exists": [col in sold.columns for col in outlier_fields]
})

field_check

,field,exists
0,ClosePrice,True
1,LivingArea,True
2,DaysOnMarket,True


## Apply Business-Rule Data Quality Flags

Before applying IQR, I created business-rule flags for clearly invalid values. These records are not simply deleted from the original dataset; they are flagged first so the full dataset is preserved.

In [25]:
sold["invalid_close_price_flag"] = sold["ClosePrice"].isna() | (sold["ClosePrice"] <= 0)
sold["invalid_living_area_flag"] = sold["LivingArea"].isna() | (sold["LivingArea"] <= 0)
sold["invalid_days_on_market_flag"] = sold["DaysOnMarket"].isna() | (sold["DaysOnMarket"] < 0)

business_rule_summary = pd.DataFrame({
    "flag": [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag"
    ],
    "flagged_records": [
        sold["invalid_close_price_flag"].sum(),
        sold["invalid_living_area_flag"].sum(),
        sold["invalid_days_on_market_flag"].sum()
    ]
})

business_rule_summary["flagged_pct"] = business_rule_summary["flagged_records"] / len(sold)

business_rule_summary

,flag,flagged_records,flagged_pct
0,invalid_close_price_flag,3,0.000008
1,invalid_living_area_flag,373,0.000938
2,invalid_days_on_market_flag,46,0.000116


## Calculate IQR Thresholds

The IQR method defines outliers as values below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR. Thresholds are calculated separately for ClosePrice, LivingArea, and DaysOnMarket.

In [26]:
iqr_thresholds = []

for col in outlier_fields:
    valid_series = sold[col].dropna()
    
    q1 = valid_series.quantile(0.25)
    q3 = valid_series.quantile(0.75)
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    iqr_thresholds.append({
        "field": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    })

iqr_thresholds_df = pd.DataFrame(iqr_thresholds)

iqr_thresholds_df

,field,q1,q3,iqr,lower_bound,upper_bound
0,ClosePrice,575000.0,1300000.0,725000.0,-512500.0,2387500.0
1,LivingArea,1247.0,2217.0,970.0,-208.0,3672.0
2,DaysOnMarket,8.0,48.0,40.0,-52.0,108.0


## Add IQR Outlier Flags

Instead of deleting records immediately, I added one IQR outlier flag for each key numeric field.

In [27]:
for _, row in iqr_thresholds_df.iterrows():
    col = row["field"]
    lower = row["lower_bound"]
    upper = row["upper_bound"]
    
    flag_col = f"{col.lower()}_iqr_outlier_flag"
    
    sold[flag_col] = (
        sold[col].notna() &
        ((sold[col] < lower) | (sold[col] > upper))
    )

iqr_flag_cols = [
    "closeprice_iqr_outlier_flag",
    "livingarea_iqr_outlier_flag",
    "daysonmarket_iqr_outlier_flag"
]

sold[iqr_flag_cols].sum()

closeprice_iqr_outlier_flag      29410
livingarea_iqr_outlier_flag      17540
daysonmarket_iqr_outlier_flag    30269
dtype: int64

## Create Combined Outlier Flag

The combined flag identifies records that fail any business rule or are detected as outliers by IQR in any key numeric field.

In [28]:
sold["any_business_rule_invalid_flag"] = (
    sold["invalid_close_price_flag"] |
    sold["invalid_living_area_flag"] |
    sold["invalid_days_on_market_flag"]
)

sold["any_iqr_outlier_flag"] = (
    sold["closeprice_iqr_outlier_flag"] |
    sold["livingarea_iqr_outlier_flag"] |
    sold["daysonmarket_iqr_outlier_flag"]
)

sold["remove_from_clean_analysis_flag"] = (
    sold["any_business_rule_invalid_flag"] |
    sold["any_iqr_outlier_flag"]
)

overall_flag_summary = pd.DataFrame({
    "category": [
        "Business-rule invalid records",
        "IQR outlier records",
        "Total records removed from clean analysis"
    ],
    "record_count": [
        sold["any_business_rule_invalid_flag"].sum(),
        sold["any_iqr_outlier_flag"].sum(),
        sold["remove_from_clean_analysis_flag"].sum()
    ]
})

overall_flag_summary["record_pct"] = overall_flag_summary["record_count"] / len(sold)

overall_flag_summary

,category,record_count,record_pct
0,Business-rule invalid records,422,0.001061
1,IQR outlier records,62132,0.156266
2,Total records removed from clean analysis,62349,0.156812


## Create Clean Filtered Analysis Dataset

The full dataset is preserved with flags. A separate clean analysis dataset is created by excluding records marked by the combined removal flag.

In [29]:
sold_clean_filtered = sold[~sold["remove_from_clean_analysis_flag"]].copy()

print(f"Full flagged dataset rows: {len(sold):,}")
print(f"Clean filtered dataset rows: {len(sold_clean_filtered):,}")
print(f"Rows removed: {len(sold) - len(sold_clean_filtered):,}")
print(f"Removal percentage: {(len(sold) - len(sold_clean_filtered)) / len(sold):.2%}")

Full flagged dataset rows: 397,603
Clean filtered dataset rows: 335,254
Rows removed: 62,349
Removal percentage: 15.68%


## Compare Dataset Size and Median Values Before and After Filtering

This comparison shows how outlier filtering changes the dataset size and median values for key numeric fields.

In [30]:
comparison_rows = []

for col in outlier_fields:
    comparison_rows.append({
        "field": col,
        "before_row_count": sold[col].notna().sum(),
        "after_row_count": sold_clean_filtered[col].notna().sum(),
        "before_median": sold[col].median(),
        "after_median": sold_clean_filtered[col].median(),
        "median_change": sold_clean_filtered[col].median() - sold[col].median(),
        "before_mean": sold[col].mean(),
        "after_mean": sold_clean_filtered[col].mean(),
        "mean_change": sold_clean_filtered[col].mean() - sold[col].mean()
    })

before_after_comparison = pd.DataFrame(comparison_rows)

before_after_comparison

,field,before_row_count,after_row_count,before_median,after_median,median_change,before_mean,after_mean,mean_change
0,ClosePrice,397601,335254,820000.0,785000.0,-35000.0,1.185616e+06,898084.150535,-287532.209591
1,LivingArea,397374,335254,1641.0,1568.0,-73.0,1.904351e+03,1673.697779,-230.653678
2,DaysOnMarket,397603,335254,19.0,16.0,-3.0,3.733679e+01,26.375196,-10.961592


## Percentile Review Before and After Filtering

Percentile summaries help evaluate whether extreme values were reduced while preserving the typical market range.

In [31]:
before_percentiles = sold[outlier_fields].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

after_percentiles = sold_clean_filtered[outlier_fields].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

before_percentiles["dataset"] = "before_filtering"
after_percentiles["dataset"] = "after_filtering"

percentile_comparison = pd.concat([
    before_percentiles,
    after_percentiles
]).reset_index().rename(columns={"index": "field"})

percentile_comparison

,field,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,dataset
0,ClosePrice,397601.0,1.185616e+06,5.922380e+06,0.00,203000.0,340000.0,575000.0,820000.0,1300000.0,2850000.0,5550000.0,989500000.0,before_filtering
1,LivingArea,397374.0,1.904351e+03,2.701781e+04,0.00,604.0,839.0,1247.0,1641.0,2217.0,3558.0,5280.0,17021321.0,before_filtering
2,DaysOnMarket,397603.0,3.733679e+01,5.353925e+01,-288.00,0.0,1.0,8.0,19.0,48.0,131.0,229.0,12430.0,before_filtering
3,ClosePrice,335254.0,8.980842e+05,4.599077e+05,1.15,210000.0,340000.0,563770.0,785000.0,1150000.0,1850000.0,2250000.0,2387500.0,after_filtering
4,LivingArea,335254.0,1.673698e+03,6.268312e+02,1.00,605.0,828.0,1212.0,1568.0,2035.0,2908.0,3408.0,3672.0,after_filtering
5,DaysOnMarket,335254.0,2.637520e+01,2.558334e+01,0.00,0.0,1.0,7.0,16.0,39.0,83.0,102.0,108.0,after_filtering


## Save Week 7 Outputs

The Week 7 deliverable requires both a full flagged dataset and a clean filtered dataset. Summary tables are also saved for documentation.

In [32]:
full_flagged_output = OUTPUT_DIR / "sold_week7_full_flagged_dataset.csv"
clean_filtered_output = OUTPUT_DIR / "sold_week7_clean_filtered_dataset.csv"
iqr_thresholds_output = OUTPUT_DIR / "week7_iqr_thresholds.csv"
business_rule_output = OUTPUT_DIR / "week7_business_rule_summary.csv"
overall_flag_output = OUTPUT_DIR / "week7_overall_flag_summary.csv"
before_after_output = OUTPUT_DIR / "week7_before_after_comparison.csv"
percentile_comparison_output = OUTPUT_DIR / "week7_percentile_comparison.csv"

sold.to_csv(full_flagged_output, index=False)
sold_clean_filtered.to_csv(clean_filtered_output, index=False)
iqr_thresholds_df.to_csv(iqr_thresholds_output, index=False)
business_rule_summary.to_csv(business_rule_output, index=False)
overall_flag_summary.to_csv(overall_flag_output, index=False)
before_after_comparison.to_csv(before_after_output, index=False)
percentile_comparison.to_csv(percentile_comparison_output, index=False)

print("Saved Week 7 outputs:")
print(full_flagged_output)
print(clean_filtered_output)
print(iqr_thresholds_output)
print(business_rule_output)
print(overall_flag_output)
print(before_after_output)
print(percentile_comparison_output)

Saved Week 7 outputs:
/Users/amyliu/Desktop/IDX/data/week7/sold_week7_full_flagged_dataset.csv
/Users/amyliu/Desktop/IDX/data/week7/sold_week7_clean_filtered_dataset.csv
/Users/amyliu/Desktop/IDX/data/week7/week7_iqr_thresholds.csv
/Users/amyliu/Desktop/IDX/data/week7/week7_business_rule_summary.csv
/Users/amyliu/Desktop/IDX/data/week7/week7_overall_flag_summary.csv
/Users/amyliu/Desktop/IDX/data/week7/week7_before_after_comparison.csv
/Users/amyliu/Desktop/IDX/data/week7/week7_percentile_comparison.csv
